# MiniMind: learn training from random initialization to inference

This notebook calls the repository's native PyTorch trainer scripts through `colab/minimind_colab.py`. Run each cell in order. First complete the small `micro` path; only then enable the full mini-data `zero` path.

In [29]:
# Set these to the branch that contains this colab/ directory.
import os

REPOSITORY_URL = 'https://github.com/wangzheng422/minimind.git'
REPOSITORY_REF = 'wzh-main'
ROOT = '/content/minimind'
COLAB_PYTHON = f'{ROOT}/.colab-venv/bin/python'
COLAB_RUNNER = f'{ROOT}/colab/minimind_colab.py'
REQUIRE_A100 = False  # Set True only when you specifically need an A100.
PREFLIGHT_ACCELERATOR_ARG = '--require-a100' if REQUIRE_A100 else ''
RUN_ZERO_PROFILE = True
ZERO_BATCH_SIZE = 8  # Conservative default for 22 GB-class GPUs such as L4.
RUN_TOKENIZER_EXPERIMENT = True
USE_GOOGLE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/colab/minimind/'

os.environ.update({
    'REPOSITORY_URL': REPOSITORY_URL,
    'REPOSITORY_REF': REPOSITORY_REF,
    'ROOT': ROOT,
    'COLAB_RUNNER': COLAB_RUNNER,
})


In [25]:
%%bash
set -euo pipefail

if [ -d "$ROOT/.git" ]; then
  git -C "$ROOT" fetch --depth 1 origin "$REPOSITORY_REF"
  git -C "$ROOT" checkout --detach FETCH_HEAD
elif [ -e "$ROOT" ]; then
  printf '%s\n' "ERROR: $ROOT exists but is not a Git checkout" >&2
  exit 1
else
  git clone \
    --depth 1 \
    --branch "$REPOSITORY_REF" \
    "$REPOSITORY_URL" \
    "$ROOT"
fi

cd "$ROOT"
python "$COLAB_RUNNER" --root "$ROOT" setup


{
  "venv_python": "/content/minimind/.colab-venv/bin/python",
  "next": "/content/minimind/.colab-venv/bin/python colab/minimind_colab.py preflight"
}


From https://github.com/wangzheng422/minimind
 * branch            wzh-main   -> FETCH_HEAD
 + 60d357d...280088c wzh-main   -> origin/wzh-main  (forced update)
any of your branches:

  60d357d fix: pass dataset paths as strings in Colab runner

If you want to keep it by creating a new branch, this may be a good time
to do so with:

 git branch <new-branch-name> 60d357d

HEAD is now at 280088c fix: stabilize Colab inference metrics and device selection


In [15]:
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} preflight {PREFLIGHT_ACCELERATOR_ARG}


{
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "repo": "/content/minimind",
  "torch": "2.13.0+cu130",
  "gpu": {
    "name": "NVIDIA L4",
    "memory_gib": 22.03,
    "bf16_supported": true,
    "cuda_runtime": "13.0"
  },
  "ram_gib": 52.96,
  "disk_free_gib": 57.41
}


## 1. Tokenizer, model tensors, and next-token labels

The next cells expose the same tokenizer, causal logits, and shifted labels that the trainer uses. `loss: true` means the following token is a supervised target.

In [21]:
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} lesson tokenizer --text '语言模型通过预测下一个 token 来学习文本。'
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} lesson model --profile micro


{
  "text": "语言模型通过预测下一个 token 来学习文本。",
  "input_ids": [
    1282,
    1135,
    642,
    2202,
    504,
    466,
    364,
    110,
    327,
    256,
    509,
    768,
    1896,
    302
  ],
  "raw_tokens": [
    "è¯Ńè¨Ģ",
    "æ¨¡åŀĭ",
    "éĢļè¿ĩ",
    "é¢Ħæµĭ",
    "ä¸ĭ",
    "ä¸Ģä¸ª",
    "Ġto",
    "k",
    "en",
    "Ġ",
    "æĿ¥",
    "åŃ¦ä¹ł",
    "æĸĩæľ¬",
    "ãĢĤ"
  ],
  "decoded": "语言模型通过预测下一个 token 来学习文本。",
  "special_ids": {
    "bos": 1,
    "eos": 2,
    "pad": 0
  },
  "chat_template": "<|im_start|>user\n语言模型通过预测下一个 token 来学习文本。<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
}
{
  "profile": "micro",
  "input_ids_shape": [
    1,
    8
  ],
  "embedding_shape": [
    1,
    8,
    256
  ],
  "q_projection_weight_shape": [
    256,
    256
  ],
  "k_projection_weight_shape": [
    128,
    256
  ],
  "v_projection_weight_shape": [
    128,
    256
  ],
  "logits_shape": [
    1,
    8,
    6400
  ],
  "parameters": 4983296,
  "embedding_and_lm_head_share_sto

In [22]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} restore --drive-dir {DRIVE_DIR} --allow-missing
    import subprocess, threading, time
    from pathlib import Path
    Path(ROOT, 'logs').mkdir(exist_ok=True)
    def backup_loop():
        while True:
            with open(f'{ROOT}/logs/drive-backup.log', 'a') as log:
                subprocess.run([COLAB_PYTHON, COLAB_RUNNER, '--root', ROOT, 'backup', '--drive-dir', DRIVE_DIR], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, check=False)
            time.sleep(600)
    threading.Thread(target=backup_loop, daemon=True).start()
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} download --all
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} make-micro --rows 2048
if RUN_TOKENIZER_EXPERIMENT:
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} tokenizer-experiment
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} lesson pretrain-labels --profile micro --rows 24
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} lesson sft-labels --profile micro --rows 48


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{"restore": "/content/drive/MyDrive/colab/minimind/minimind-colab", "files_copied": 1}
downloaded pretrain_t2t_mini.jsonl: /content/minimind/dataset/pretrain_t2t_mini.jsonl
downloaded sft_t2t_mini.jsonl: /content/minimind/dataset/sft_t2t_mini.jsonl
wrote 2048 records to /content/minimind/dataset/pretrain_t2t_micro.jsonl
wrote 2048 records to /content/minimind/dataset/sft_t2t_micro.jsonl

[
  {
    "position": 0,
    "input": "<|im_start|>",
    "next_token_target": "给我",
    "loss": true
  },
  {
    "position": 1,
    "input": "给我",
    "next_token_target": "生成",
    "loss": true
  },
  {
    "position": 2,
    "input": "生成",
    "next_token_target": "一",
    "loss": true
  },
  {
    "position": 3,
    "input": "一",
    "next_token_target": "首",
    "loss": true
  },
  {
    "position": 4,
    "input": "首",
    "next_token_target": "有关",
    "loss": true
  

## 2. Execute one real optimizer update

This is not a mock: it runs forward, cross-entropy, backward, gradient clipping, AdamW, and a second loss measurement on the same batch.

In [26]:
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} one-step --profile micro --stage pretrain
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} train --profile micro --stage pretrain --resume
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} infer --profile micro --stage pretrain --prompt '人工智能是' --max-new-tokens 80
if USE_GOOGLE_DRIVE:
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} backup --drive-dir {DRIVE_DIR}


{
  "stage": "pretrain",
  "profile": "micro",
  "device": "cuda",
  "batch_shape": [
    8,
    128
  ],
  "logits_shape": [
    8,
    128,
    6400
  ],
  "loss_before": 8.827217102050781,
  "grad_norm_before_clip": 3.1875479221343994,
  "loss_after_same_batch": 8.594398498535156,
  "peak_gpu_memory_gib": 0.186
}
RUN: /content/minimind/.colab-venv/bin/python /content/minimind/trainer/train_pretrain.py --save_dir /content/minimind/out --save_weight learn_micro_pretrain --epochs 1 --batch_size 8 --accumulation_steps 1 --max_seq_len 128 --data_path /content/minimind/dataset/pretrain_t2t_micro.jsonl --hidden_size 256 --num_hidden_layers 4 --dtype bfloat16 --num_workers 2 --from_resume 1 --use_compile 0 --from_weight none
Model Params: 4.98M
Trainable Params: 4.983M
Epoch [1/1]: 跳过前256个step，从step 257开始
这根据为我一个篇关于我“
一个篇句子。
{"backup": "/content/drive/MyDrive/colab/minimind/minimind-colab", "files_copied": 5}


In [27]:
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} one-step --profile micro --stage sft
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} train --profile micro --stage sft --resume
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} infer --profile micro --stage sft --prompt '请解释什么是自注意力机制。' --max-new-tokens 128
if USE_GOOGLE_DRIVE:
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} backup --drive-dir {DRIVE_DIR}


{
  "stage": "sft",
  "profile": "micro",
  "device": "cuda",
  "batch_shape": [
    8,
    128
  ],
  "logits_shape": [
    8,
    128,
    6400
  ],
  "loss_before": 6.904371738433838,
  "grad_norm_before_clip": 2.446578025817871,
  "loss_after_same_batch": 6.780662536621094,
  "peak_gpu_memory_gib": 0.186
}
RUN: /content/minimind/.colab-venv/bin/python /content/minimind/trainer/train_full_sft.py --save_dir /content/minimind/out --save_weight learn_micro_sft --epochs 1 --batch_size 8 --accumulation_steps 1 --max_seq_len 128 --data_path /content/minimind/dataset/sft_t2t_micro.jsonl --hidden_size 256 --num_hidden_layers 4 --dtype bfloat16 --num_workers 2 --from_resume 1 --use_compile 0 --from_weight learn_micro_pretrain
Model Params: 4.98M
Trainable Params: 4.983M
Epoch:[1/1](100/256), loss: 6.5366, logits_loss: 6.5366, aux_loss: 0.0000, lr: 0.00000702, epoch_time: 0.0min
Epoch:[1/1](200/256), loss: 6.3404, logits_loss: 6.3404, aux_loss: 0.0000, lr: 0.00000202, epoch_time: 0.0min
Epoch

## 3. Optional: persist outputs in Google Drive

When enabled before data preparation, Drive is restored first, backed up every 10 minutes, and copied again after each stage. Training always remains on Colab's local disk.

In [ ]:
if USE_GOOGLE_DRIVE:
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} backup --drive-dir {DRIVE_DIR}


## 4. Optional: MiniMind Zero reproduction

Set `RUN_ZERO_PROFILE = True` only after the micro route succeeds. This uses the complete official mini JSONL files and the repository's 768-hidden, 8-layer configuration. `ZERO_BATCH_SIZE = 8` is the conservative L4 learning variant; because it lowers the effective batch from the default profile, it is not an exact reference reproduction. Increase it only after observing stable GPU memory usage.

In [ ]:
if RUN_ZERO_PROFILE:
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} lesson model --profile zero
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} train --profile zero --stage pretrain --batch-size {ZERO_BATCH_SIZE} --resume
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} infer --profile zero --stage pretrain --prompt '机器学习是一种' --max-new-tokens 128
    if USE_GOOGLE_DRIVE:
        !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} backup --drive-dir {DRIVE_DIR}
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} train --profile zero --stage sft --batch-size {ZERO_BATCH_SIZE} --resume
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} infer --profile zero --stage sft --prompt '请用通俗语言解释 Transformer。' --max-new-tokens 256
    if USE_GOOGLE_DRIVE:
        !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} backup --drive-dir {DRIVE_DIR}


{
  "profile": "zero",
  "input_ids_shape": [
    1,
    8
  ],
  "embedding_shape": [
    1,
    8,
    768
  ],
  "q_projection_weight_shape": [
    768,
    768
  ],
  "k_projection_weight_shape": [
    384,
    768
  ],
  "v_projection_weight_shape": [
    384,
    768
  ],
  "logits_shape": [
    1,
    8,
    6400
  ],
  "parameters": 63912192,
  "embedding_and_lm_head_share_storage": true
}
RUN: /content/minimind/.colab-venv/bin/python /content/minimind/trainer/train_pretrain.py --save_dir /content/minimind/out --save_weight learn_zero_pretrain --epochs 1 --batch_size 8 --accumulation_steps 8 --max_seq_len 768 --data_path /content/minimind/dataset/pretrain_t2t_mini.jsonl --hidden_size 768 --num_hidden_layers 8 --dtype bfloat16 --num_workers 2 --from_resume 1 --use_compile 0 --from_weight none
Model Params: 63.91M
Trainable Params: 63.912M

Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 70908 examples [00:00, 647968.37 examples/s]
Generating trai

: 